[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/yellow/notebooks/yellow_baseline_models.ipynb)

# Training baseline models for ACE1 inhibition

**Yellow group · Hypertension**

This notebook trains the first machine learning models on the ACE1 data curated in the
first notebook: one that says whether a molecule inhibits the enzyme, and one that
predicts how strongly. These simple models are the baseline that any better model has
to beat, and they are the models the group will later point at indoles and xanthones.

## What you will do

- Plot how the data is distributed: actives against inactives, potency and molecular size.
- Turn every molecule into a fingerprint, a row of numbers a model can read.
- Train a random forest classifier and a random forest regressor, and measure how good they are.
- Compare a random split of the data with a scaffold split, using 5-fold cross-validation.

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "yellow"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. Load the curated data

The first notebook turned four sources of ACE1 data into a single table, and the group
uploaded it to its Drive folder. It is copied into `data/` in this repository, so it is
already here.

Each row is one molecule. The columns we need are:

- `smiles`, the molecule itself.
- `activity`, the label: 1 for an inhibitor, 0 for not. We use it for **classification**.
- `pactivity`, how strongly it inhibits, where higher means more potent. We use it for
  **regression**.
- `evidence`, where the label came from: a real measurement, or only a limit such as
  "no inhibition at 100 uM".

In [ ]:
import numpy as np
import pandas as pd
import stylia
from scripts import modelling

RANDOM_SEED = 42
THRESHOLD = 6.0  # pActivity 6 is the 1 uM cutoff used to label the molecules

curated = pd.read_csv("data/ace_human_curated.csv")
print(f"{len(curated):,} molecules")
curated[["inchikey", "smiles", "pactivity", "activity", "evidence"]].head()

Not every molecule has a measured number. Some are known to be inactive only because a
paper said so, or because the experiment never reached 50% inhibition. Those are perfectly
good **labels**, but they are not usable **values**, so they can train the classifier and
not the regressor.

In [ ]:
curated["evidence"].value_counts().to_frame("molecules")

## 2. Look at the data

Before training anything, look at what the model will learn from. A model can only be as
good as its data, and a few plots often reveal problems that no metric will show later.

We start with the **class balance**: how many molecules inhibit ACE1 and how many do not.

In [ ]:
stylia.set_format("slide")
stylia.set_style("ersilia")
nc = stylia.NamedColors()

counts = curated["activity"].value_counts().rename({0: "inactive", 1: "active"})
counts = counts.reindex(["inactive", "active"])  # the order the colours below assume

fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
ax.bar(counts.index, counts.values, color=[nc.blue, nc.yellow])
stylia.label(ax, xlabel="", ylabel="Molecules",
             title=f"{counts['active'] / counts.sum():.0%} of the molecules are active")

Two thirds of the molecules are active. That is **class imbalance**, and it matters in two
ways.

First, accuracy becomes misleading: a model that called every single molecule active would
already be right two thirds of the time while having learned nothing. We will use metrics
that do not fall for this.

Second, it is not an accident. Nobody publishes a paper about a molecule that does nothing,
so databases are full of compounds that worked. The model therefore sees a world with far
more inhibitors in it than the real world has.

> **Note:** The project plan lists "not enough data" as a risk for this project. With about
> a thousand molecules that risk is real, and it is the reason we lean on cross-validation
> in section 6 rather than trusting a single split.

Next, the **potency distribution**. The dashed line is the cutoff that separates the two
classes, at 1 uM.

In [ ]:
measured = curated[curated["evidence"] == "measured"]
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
ax.hist(measured["pactivity"], bins=40, range=(2.5, 11), color=nc.yellow)
ax.axvline(THRESHOLD, color=nc.pink, linestyle="--")
stylia.label(ax, xlabel="pActivity", ylabel="Molecules",
             title=f"Measured potency (median {measured['pactivity'].median():.2f})")

Size can give a model an easy shortcut. If the actives were simply bigger than the
inactives, the model could learn "big means active" and nothing about chemistry. We compute
the **molecular weight** of every molecule to check.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

curated["mw"] = [Descriptors.MolWt(Chem.MolFromSmiles(smi)) for smi in curated["smiles"]]
curated.groupby("activity")["mw"].describe().round(0)

The histogram shows the two classes side by side, with the same colours as the bar chart
above.

In [ ]:
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
for label, color, name in [(0, nc.blue, "inactive"), (1, nc.yellow, "active")]:
    weights = curated.loc[curated["activity"] == label, "mw"]
    ax.hist(weights, bins=40, range=(0, 1000), histtype="stepfilled", alpha=0.6,
            color=color, label=f"{name} (median {weights.median():.0f})")
ax.legend()
stylia.label(ax, xlabel="Molecular weight (g/mol)", ylabel="Molecules",
             title="Molecular weight of actives and inactives")

The two distributions sit almost on top of each other: the medians differ by about ten
g/mol, which is nothing. So size carries no information about whether a molecule inhibits
ACE1, and the model cannot take that shortcut. It has to use the actual chemistry.

Last, the **chemical series**. Medicinal chemists usually make many variations of one core
structure. That core is called the **scaffold** (or Bemis-Murcko scaffold): the rings of a
molecule and the chains connecting them, with every side group removed. We will need the
scaffolds in section 6, so we compute them now.

In [ ]:
curated["scaffold"] = modelling.murcko_scaffolds(curated["smiles"])
series = curated["scaffold"].value_counts()
print(f"{len(curated):,} molecules share {len(series):,} scaffolds")
print(f"{(series == 1).sum():,} scaffolds appear only once")
series.head(5).to_frame("molecules")

This dataset is unusually **diverse**: a quarter of the molecules are the only example of
their scaffold. Compare that with a dataset built by optimising one chemical series, where
a handful of scaffolds would cover almost everything.

Diversity is good news and bad news. Good, because the model sees many kinds of molecule.
Bad, because with about a thousand molecules spread over hundreds of series, it sees very
few examples of each.

> **Exercise:** Paste the most common scaffold into a structure viewer (for example
> https://molview.org). Many ACE1 inhibitors share a core because they were designed from
> the same starting point. Does the one you see look like the drugs the group found in the
> curation notebook?

## 3. Turn molecules into numbers

A model cannot read a SMILES string. It needs every molecule written as the same fixed list
of numbers, called **features**. Turning molecules into features is called **featurisation**.

We use a **Morgan fingerprint** (also called ECFP4). For every atom it looks at the small
fragment around it, up to two bonds away, and switches on one of 2,048 bits for that
fragment. Two molecules sharing many fragments share many bits, so similar molecules get
similar fingerprints.

In [ ]:
X_class = modelling.morgan_fingerprints(curated["smiles"], radius=2, n_bits=2048)
y_class = curated["activity"].values
print(f"feature matrix: {X_class.shape[0]:,} molecules x {X_class.shape[1]:,} bits")
print(f"on average {X_class.sum(axis=1).mean():.0f} bits are switched on per molecule")

For regression we keep only the molecules with a real measured value, since a limit such as
"no inhibition at 100 uM" is not a number a model can learn to predict. Those molecules are
a subset of the ones above, so we pick out their rows rather than featurising again.

In [ ]:
is_measured = (curated["evidence"] == "measured") & curated["pactivity"].notna()
X_reg = X_class[is_measured.values]
y_reg = curated.loc[is_measured, "pactivity"].values
scaffolds_reg = curated.loc[is_measured, "scaffold"].values
print(f"{len(y_class):,} molecules for classification")
print(f"{len(y_reg):,} molecules for regression")

## 4. Train a classifier

To know whether a model works we must test it on molecules it has **never seen**. So we hold
back 20% of the molecules as a **test set** and train on the other 80%. The split is
stratified, meaning both parts keep the same share of actives.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_class, y_class, test_size=0.2, stratify=y_class, random_state=RANDOM_SEED)
print(f"training set: {len(y_train):,} molecules, {y_train.mean():.1%} active")
print(f"test set:     {len(y_test):,} molecules, {y_test.mean():.1%} active")

A **random forest** is a collection of decision trees. Each tree asks a series of yes-or-no
questions about the bits ("does the molecule contain this fragment?"), and the forest
averages the answers of all its trees.

`class_weight="balanced"` tells the model to care as much about the smaller class as the
larger one. With two thirds of the data active, that matters here.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

classifier = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                    n_jobs=-1, random_state=RANDOM_SEED)
classifier.fit(X_train, y_train)
proba = classifier.predict_proba(X_test)[:, 1]
print(f"predicted the probability of being active for {len(proba):,} test molecules")

The model gives each test molecule a **probability** of being active. We call it active when
that probability is 0.5 or more, and compare with the real labels. The metrics mean:

- **ROC-AUC**: how often an active molecule scores higher than an inactive one. 0.5 is a coin
  toss, 1 is perfect.
- **PR-AUC**: how precise the model stays as it finds more of the actives. Careful with this
  one: a model guessing at random scores the share of actives, which is about 0.66 here, not
  0.5. Only the amount above 0.66 is real skill.
- **Balanced accuracy**: the average of the share of actives and the share of inactives it
  gets right. Unlike plain accuracy, always calling "active" scores 0.5.
- **Precision**: of the molecules it calls active, how many really are.
- **Recall**: of the real actives, how many it finds.

In [ ]:
scores = modelling.classification_metrics(y_test, proba)
print(f"always guessing 'active' would give PR-AUC {y_test.mean():.3f}")
pd.Series(scores).round(3).to_frame("test set")

The **ROC curve** shows the trade-off behind ROC-AUC. Moving along the curve lowers the
probability needed to call a molecule active: the model finds more actives (up) but also
calls more inactives active by mistake (right). The diagonal is a coin toss.

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_test, proba)
fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
ax.plot(fpr, tpr, color=nc.yellow)
ax.plot([0, 1], [0, 1], color=nc.gray, linestyle=":")
stylia.label(ax, xlabel="False positive rate", ylabel="True positive rate",
             title="ROC curve (test set)")

The **confusion matrix** counts the four possible outcomes: actives called active, actives
missed, inactives called inactive, and inactives wrongly called active.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
ConfusionMatrixDisplay.from_predictions(
    y_test, (proba >= 0.5).astype(int), display_labels=["inactive", "active"],
    cmap="YlOrBr", colorbar=False, ax=ax)
ax.grid(False)
stylia.label(ax, xlabel="Predicted", ylabel="Real", title="Confusion matrix (test set)")

> **Note:** The test set holds only about 200 molecules, so each cell of this matrix rests on
> a small count and would move if we picked a different split. That is exactly why section 6
> repeats the whole thing five times.

> **Exercise:** Change the 0.5 in the cell above to 0.3 and then to 0.7. How do the four
> numbers move? The group plans to screen natural products and test the hits in the
> laboratory, so which mistake costs more: missing an inhibitor, or sending a useless
> compound for testing?

## 5. Train a regressor

Now the harder question: not just active or not, but **how** active. The model predicts the
pActivity itself. We split the measured molecules 80/20 in the same way (no stratification
this time, as there are no classes).

In [ ]:
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_SEED)
print(f"training set: {len(yr_train):,} molecules")
print(f"test set:     {len(yr_test):,} molecules")

A random forest regressor works like the classifier, but each tree predicts a number and the
forest averages them. `max_features="sqrt"` lets each question in a tree consider only a
random handful of the 2,048 bits, which makes the trees more varied and training faster.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

regressor = RandomForestRegressor(n_estimators=100, max_features="sqrt",
                                  n_jobs=-1, random_state=RANDOM_SEED)
regressor.fit(Xr_train, yr_train)
predicted = regressor.predict(Xr_test)
print(f"predicted the pActivity of {len(predicted):,} test molecules")

The regression metrics mean:

- **R2**: how much of the spread in the real values the model explains. 0 means no better
  than always guessing the average, 1 is perfect.
- **RMSE** and **MAE**: the typical error, in pActivity units. An error of 1 means the
  prediction is off by a factor of ten in concentration.
- **Spearman**: whether the model puts the molecules in the right order, from weakest to most
  potent, even if the numbers themselves are off.

In [ ]:
pd.Series(modelling.regression_metrics(yr_test, predicted)).round(3).to_frame("test set")

Each point below is one test molecule. A perfect model would put every point on the
diagonal.

In [ ]:
fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
ax.scatter(yr_test, predicted, color=nc.yellow, alpha=0.5)
ax.plot([3, 11], [3, 11], color=nc.gray, linestyle=":")
stylia.label(ax, xlabel="Measured pActivity", ylabel="Predicted pActivity",
             title="Regression (test set)")

Notice how the points flatten out: the most potent molecules are predicted too low and the
weakest too high. Random forests average many trees, so they are pulled towards the middle
and rarely predict extreme values. That is worth remembering when the model is later used to
rank natural products: it is better at saying "this one is stronger than that one" than at
putting an exact number on either.

## 6. Random or scaffold split: a fairer test

So far we split the molecules at random. Because molecules come in series, a random split
puts close relatives of many test molecules into the training set. The model has then seen
something very similar before, and the test is easy.

That is not how this model will be used. The group wants to screen **indoles and xanthones**,
natural products that look nothing like the peptide-derived drugs that dominate ACE1 data. A
fairer test is a **scaffold split**: every molecule of a series goes to the same side, so the
test molecules have scaffolds the model has never seen.

We also switch from one split to **5-fold cross-validation**: the data is cut into five parts,
and the model is trained five times, each time tested on a different part. With a dataset
this small that matters, because a single split is easily a lucky one.

### 6.1 Classification

`StratifiedKFold` makes five random folds. `StratifiedGroupKFold` makes five folds where each
scaffold stays in a single fold. Both keep the share of actives roughly equal across folds.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold

random_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
scaffold_folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_class = pd.concat({
    "random": modelling.cross_validate(classifier, X_class, y_class, random_folds),
    "scaffold": modelling.cross_validate(classifier, X_class, y_class, scaffold_folds,
                                         groups=curated["scaffold"]),
}, names=["split"])
cv_class.groupby("split").agg(["mean", "std"]).round(3)

### 6.2 Regression

The same comparison for the regressor, with `KFold` and `GroupKFold`.

In [ ]:
from sklearn.model_selection import GroupKFold, KFold

cv_reg = pd.concat({
    "random": modelling.cross_validate(
        regressor, X_reg, y_reg, KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)),
    "scaffold": modelling.cross_validate(
        regressor, X_reg, y_reg, GroupKFold(n_splits=5), groups=scaffolds_reg),
}, names=["split"])
cv_reg.groupby("split").agg(["mean", "std"]).round(3)

### 6.3 Compare the two splits

Each dot is one fold and the dark line is the mean over the five folds. The further apart the
two splits are, the more the random split flattered the model.

In [ ]:
fig, axs = stylia.create_figure(1, 2)
for cv, metric in [(cv_class, "ROC-AUC"), (cv_reg, "R2")]:
    ax = axs.next()
    for i, (split, color) in enumerate([("random", nc.yellow), ("scaffold", nc.blue)]):
        fold_scores = cv.loc[split, metric]
        ax.scatter([i] * len(fold_scores), fold_scores, color=color)
        ax.hlines(fold_scores.mean(), i - 0.25, i + 0.25, color=nc.plum)
    ax.set_xticks([0, 1], ["random", "scaffold"])
    ax.set_xlim(-0.6, 1.6)
    stylia.label(ax, xlabel="Split", ylabel=metric, title=f"{metric} over 5 folds")

Both models score clearly worse on the scaffold split, and the regressor loses most. The
scaffold numbers are the ones to report and to beat, because they are the closest thing we
have to predicting molecules from a series the model has never seen, which is exactly what
indoles and xanthones will be.

Notice also how far apart the five dots are within a split. With about a thousand molecules,
the difference between a good fold and a bad one is almost as large as the difference between
the two kinds of split. Any future model has to beat this baseline by more than that spread
before the improvement is believable.

> **Exercise:** Change the fingerprint in section 3, for example to `radius=3` or
> `n_bits=1024`, and run the notebook again from there. Does the scaffold split score change
> by more than the spread between folds? If not, the change did nothing.

> **Exercise:** Remove `class_weight="balanced"` from the classifier and compare. Which
> metrics move, and which barely notice? Recall and balanced accuracy should react more than
> ROC-AUC.

## Summary

- You looked at the data before modelling: two thirds of the molecules are active, they are
  spread over hundreds of scaffolds with a quarter of them unique, and actives and inactives
  are the same size, so the model cannot take that shortcut.
- You turned each molecule into a 2,048-bit Morgan fingerprint and trained a random forest
  classifier (inhibitor or not) and a random forest regressor (pActivity).
- With a random split the classifier reaches a ROC-AUC of about 0.90 and the regressor an R2
  of about 0.58. With a scaffold split both drop, to about 0.84 and 0.40: that is the fairer
  estimate for molecules from new series.
- With only about a thousand molecules the fold-to-fold spread is wide, so treat small
  differences between models with suspicion.

**Next:** check whether indoles and xanthones fall inside the applicability domain of these
models before trusting any prediction made on them.